In [ ]:
import json
import math
import numpy as np
import os
import pandas as pd

from IPython.display import display
from collections import defaultdict
from sklearn import metrics

import matplotlib.pyplot as plt
%config InlineBackend.figure_format = 'retina'

### Utility functions

In [ ]:
def merge_data(df_pairs, df_similarity, is_pos):
    df_pairs = df_pairs.merge(
        df_similarity,
        how='left',
        left_on=['idb_path_1', 'fva_1', 'idb_path_2', 'fva_2'],
        right_on=['idb_path_1', 'fva_1', 'idb_path_2', 'fva_2'])

    if is_pos:
        # If positive pairs, the perfect similarity is 1
        df_pairs['gt'] = [1] * df_pairs.shape[0]
    else:
        # if negative pairs, the perfect similarity is 0
        df_pairs['gt'] = [-1] * df_pairs.shape[0]

    return df_pairs

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from sklearn import metrics
from sklearn.metrics import roc_auc_score

def plot_sim_and_roc(df_pos, df_neg, test_name, output_dir):
    result_list = []

    # Rename columns (as before)
    df_pos.rename(columns={'db_type_x': 'db_type'}, inplace=True)
    df_neg.rename(columns={'db_type_x': 'db_type'}, inplace=True)

    task_list = sorted(set(df_pos['db_type']))
    n_tasks = len(task_list)

    # Choose subplot layout
    plt_height = 10
    plt_width = 1
    if n_tasks == 3:
        plt_height = 3
        plt_width = 3
    elif n_tasks == 4:
        plt_height = 3
        plt_width = 4

    # ─────────────────────────────────────────────────────────────────────
    # 1) Create ROC figure
    nrows = int(np.ceil(n_tasks / plt_width))
    ncols = plt_width

    fig_auc, axs = plt.subplots(
        nrows,
        ncols,
        figsize=(10, plt_height),
        squeeze=False
    )

    sim_list = []
    labels_list = []

    for idx, task in enumerate(task_list):
        row = idx // plt_width
        col = idx % plt_width

        df_pos_task = df_pos[df_pos['db_type'] == task]
        df_neg_task = df_neg[df_neg['db_type'] == task]

        # ---- for boxplot later ----
        sim_list.append(df_pos_task['sim'])
        sim_list.append(df_neg_task['sim'])
        labels_list.append(f"Pos (task: {task})")
        labels_list.append(f"Neg (task: {task})")

        # ---- Extract prediction + ground truth ----
        pred_list = list(df_pos_task['sim'].values) + list(df_neg_task['sim'].values)
        gt_list   = list(df_pos_task['gt'].values)  + list(df_neg_task['gt'].values)

        gt_arr = np.array(gt_list)
        pred_arr = np.array(pred_list)

        # ---- Remove NaN predictions ----
        mask = ~np.isnan(pred_arr)
        gt_arr_clean = gt_arr[mask]
        pred_arr_clean = pred_arr[mask]

        # ---- Handle single-class / empty cases ----
        if len(gt_arr_clean) == 0 or len(np.unique(gt_arr_clean)) < 2:
            roc_auc = 0.5
            fpr = np.array([0,1])
            tpr = np.array([0,1])
        else:
            roc_auc = roc_auc_score(gt_arr_clean, pred_arr_clean)
            fpr, tpr, _ = metrics.roc_curve(gt_arr_clean, pred_arr_clean)

        result_list.append([f"{task:>20s}", f"{roc_auc:0.2f}"])

        # ---- Plot ROC curve ----
        ax = axs[row][col]
        ax.plot(fpr, tpr, linewidth=1.0, label=f"AUC={roc_auc:0.2f}")
        ax.set_xlim([0, 1])
        ax.set_ylim([0, 1])
        ax.set_xlabel("FPR")
        ax.set_ylabel("TPR")
        ax.set_title(f"Task: {task}")
        ax.legend(loc="lower right", fontsize="small")

    # hide unused subplots
    total_plots = nrows * ncols
    if total_plots > n_tasks:
        for extra in range(n_tasks, total_plots):
            erow = extra // plt_width
            ecol = extra % plt_width
            axs[erow][ecol].axis('off')

    fig_auc.tight_layout()
    roc_path = os.path.join(output_dir, f"{test_name}_roc.pdf")
    fig_auc.savefig(roc_path, dpi=300)
    plt.close(fig_auc)

    # ─────────────────────────────────────────────────────────────────────
    # 2) Boxplot
    fig_bplot, ax_b = plt.subplots(figsize=(10, plt_height))

    bplot = ax_b.boxplot(
        x=sim_list[::-1],
        labels=labels_list[::-1],
        showfliers=False,
        patch_artist=True,
        vert=False
    )
    ax_b.set_title("Similarity distribution for positive and negative pairs")

    for idx_patch, patch in enumerate(bplot['boxes']):
        if idx_patch % 2 == 1:
            patch.set_facecolor('lightblue')

    fig_bplot.tight_layout()
    box_path = os.path.join(output_dir, f"{test_name}_boxplot.pdf")
    fig_bplot.savefig(box_path, dpi=300)
    plt.close(fig_bplot)

    return result_list


In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

def compute_auc_and_plot(df_pos, df_neg, results_dir, output_dir):
    results = []
    for csv_file in sorted(os.listdir(results_dir)):
        if not csv_file.endswith(".csv") or "pos_testing" not in csv_file:
            continue

        print(
            "[D] Processing\n\t{}\n\t{}".format(
                csv_file,
                csv_file.replace("pos_testing", "neg_testing")
            )
        )

        test_name = csv_file.replace("pos_testing_", "").replace(".csv", "")

        # ─────────────────────────────────────────────────────────
        # Load the positive/negative sim CSVs
        df_pos_sim = pd.read_csv(
            os.path.join(results_dir, csv_file),
            encoding="utf-8"
        )
        df_neg_sim = pd.read_csv(
            os.path.join(
                results_dir,
                csv_file.replace("pos_testing", "neg_testing")
            ),
            encoding="utf-8"
        )

        # Sanity check: no NaNs in “sim” column
        assert df_pos_sim['sim'].isna().sum() == 0
        assert df_neg_sim['sim'].isna().sum() == 0

        # ─────────────────────────────────────────────────────────
        # STEP 1: Plot & save the “sim distribution” histogram,
        #         then overlay short red horizontal ticks at all positive‐pair
        #         sim scores where one of the binary‐path columns contains "55215628".
        fig, ax = plt.subplots(figsize=(8, 5))

        # Plot the two histograms on the same axes
        df_pos_sim['sim'].hist(
            ax=ax,
            bins=200,
            alpha=0.6,
            label="Pos",
            color="C0"
        )
        df_neg_sim['sim'].hist(
            ax=ax,
            bins=200,
            alpha=0.6,
            label="Neg",
            color="C1"
        )

        # Determine current y‐axis limits
        y_min, y_max = ax.get_ylim()

        # We'll place ticks near the top, at 95% of y_max
        tick_y = y_max * 0.95

        # Small horizontal half‐width for each tick (in similarity units)
        # Since bins=200 span roughly [min, max], we can approximate:
        x_min, x_max = ax.get_xlim()
        bin_width = (x_max - x_min) / 200.0
        half_width = bin_width * 0.5  # a short dash half‐width

        # Find all rows in df_pos_sim where any string column contains "55215628"
        mask = df_pos_sim.applymap(
            lambda cell: isinstance(cell, str) and "55215628" in cell
        ).any(axis=1)

        # Extract the sim scores for those rows
        marked_sims = df_pos_sim.loc[mask, 'sim'].values

        # Draw short red horizontal ticks for each matching sim value
        for v in marked_sims:
            ax.hlines(
                y=tick_y,
                xmin=v - half_width,
                xmax=v + half_width,
                color='red',
                linewidth=2
            )

        ax.set_xlabel("Similarity")
        ax.set_ylabel("Count")
        ax.legend()

        sim_path = os.path.join(output_dir, f"{test_name}_sim.pdf")
        plt.tight_layout()
        plt.savefig(sim_path, dpi=300)
        plt.close()  # ← close after saving to avoid overplotting next iteration

        # ─────────────────────────────────────────────────────────
        # STEP 2: Print out the “5 positive pairs with lowest sim”
        #         and “5 negative pairs with highest sim”
        lowest_pos = df_pos_sim.nsmallest(5, 'sim')
        print(f"\n5 positive pairs with lowest sim for '{test_name}':")
        print(lowest_pos.to_string(index=False))
        print("-" * 80)

        highest_neg = df_neg_sim.nlargest(5, 'sim')
        print(f"5 negative pairs with highest sim for '{test_name}':")
        print(highest_neg.to_string(index=False))
        print("=" * 100 + "\n")

        # ─────────────────────────────────────────────────────────
        # STEP 3: Merge with your “df_pos”/“df_neg” metadata and plot ROC + boxplot
        df_pos_m = merge_data(df_pos, df_pos_sim, is_pos=True)
        df_neg_m = merge_data(df_neg, df_neg_sim, is_pos=False)

        tmp_list = [["title", test_name]]
        tmp_list.extend(plot_sim_and_roc(df_pos_m, df_neg_m, test_name, output_dir))
        results.append(tmp_list)

    return results


In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# ─────────────────────────────────────────────────────────
# Helpers for difficulty labeling
_DIFFICULTY_LEVELS = list("ABCDEF")

def _extract_difficulty_from_path(path: str):
    """
    From paths like:
      IDBs/Dataset-5/347-E/347-E-51850543.i64  -> 'E'
    We take the parent directory name and grab the last hyphen-delimited token.
    """
    if not isinstance(path, str):
        return None
    try:
        parent_dir = os.path.basename(os.path.dirname(path))  # e.g., '347-E'
        lvl = parent_dir.split('-')[-1].upper()
        return lvl if lvl in _DIFFICULTY_LEVELS else None
    except Exception:
        return None

def _add_difficulty_columns(df: pd.DataFrame) -> pd.DataFrame:
    """
    Adds 'diff1' and 'diff2' based on idb_path_1 / idb_path_2.
    """
    out = df.copy()
    if 'idb_path_1' in out.columns:
        out['diff1'] = out['idb_path_1'].map(_extract_difficulty_from_path)
    if 'idb_path_2' in out.columns:
        out['diff2'] = out['idb_path_2'].map(_extract_difficulty_from_path)
    return out

# ─────────────────────────────────────────────────────────
# Main function matching the signature and style of your original
def compute_auc_by_difficulty_and_plot(df_pos, df_neg, results_dir, output_dir):
    """
    Drop-in companion for Dataset-5 difficulty analysis.

    Parameters
    ----------
    df_pos : pd.DataFrame
        Positive-pair metadata (same as your original function expects for merge_data).
    df_neg : pd.DataFrame
        Negative-pair metadata (same as your original function expects for merge_data).
    results_dir : str
        Directory containing the per-model CSVs (pos_testing_*.csv and neg_testing_*.csv).
    output_dir : str
        Directory to save plots.

    Returns
    -------
    results : list
        A list of lists, mirroring your original style, where each entry corresponds to
        a (test_name, difficulty) slice:
          [
            ["title", f"{test_name} [L={lvl}]"],
            ... (items returned by plot_sim_and_roc for this slice)
          ]
    """
    os.makedirs(output_dir, exist_ok=True)
    results = []

    for csv_file in sorted(os.listdir(results_dir)):
        if not csv_file.endswith(".csv") or "pos_testing" not in csv_file:
            continue

        print(
            "[D] Processing by difficulty\n\t{}\n\t{}".format(
                csv_file,
                csv_file.replace("pos_testing", "neg_testing")
            )
        )
        test_name = csv_file.replace("pos_testing_", "").replace(".csv", "")

        # ─────────────────────────────────────────────────────────
        # Load the positive/negative sim CSVs
        pos_path = os.path.join(results_dir, csv_file)
        neg_path = os.path.join(results_dir, csv_file.replace("pos_testing", "neg_testing"))

        df_pos_sim = pd.read_csv(pos_path, encoding="utf-8")
        df_neg_sim = pd.read_csv(neg_path, encoding="utf-8")

        # Sanity check: no NaNs in “sim” column
        assert df_pos_sim['sim'].isna().sum() == 0
        assert df_neg_sim['sim'].isna().sum() == 0

        # ─────────────────────────────────────────────────────────
        # Merge with your metadata (reuse your existing utilities)
        df_pos_m = merge_data(df_pos, df_pos_sim, is_pos=True)
        df_neg_m = merge_data(df_neg, df_neg_sim, is_pos=False)

        # Tag difficulties from paths
        df_pos_m = _add_difficulty_columns(df_pos_m)
        df_neg_m = _add_difficulty_columns(df_neg_m)

        # ─────────────────────────────────────────────────────────
        # For each level A..F:
        #   Positives: diff1 == lvl and diff2 == lvl
        #   Negatives: diff1 == lvl and diff2 == lvl  (drop cross-level)
        for lvl in _DIFFICULTY_LEVELS:
            pos_lvl = df_pos_m[(df_pos_m['diff1'] == lvl) & (df_pos_m['diff2'] == lvl)].copy()
            neg_lvl = df_neg_m[(df_neg_m['diff1'] == lvl) & (df_neg_m['diff2'] == lvl)].copy()

            if pos_lvl.empty or neg_lvl.empty:
                print(f"[W] Skipping '{test_name}' level {lvl}: "
                      f"{'no positives' if pos_lvl.empty else ''}"
                      f"{' and ' if pos_lvl.empty and neg_lvl.empty else ''}"
                      f"{'no negatives' if neg_lvl.empty else ''}.")
                continue

            # ─────────────────────────────────────────────────────
            # STEP 1 (mirroring your diagnostics): extremes
            lowest_pos = pos_lvl.nsmallest(5, 'sim')
            print(f"\n5 positive pairs with lowest sim for '{test_name}' [L={lvl}]:")
            print(lowest_pos.to_string(index=False))
            print("-" * 80)

            highest_neg = neg_lvl.nlargest(5, 'sim')
            print(f"5 negative pairs with highest sim for '{test_name}' [L={lvl}]:")
            print(highest_neg.to_string(index=False))
            print("=" * 100 + "\n")

            # ─────────────────────────────────────────────────────
            # STEP 2: Plots/metrics via your existing plot_sim_and_roc
            # Save under a level-qualified title to avoid collisions
            level_title = f"{test_name}_Difficulty_{lvl}"
            tmp_list = [["title", f"{test_name} [L={lvl}]"]]

            # plot_sim_and_roc should both save figures to output_dir and return small metrics
            tmp_list.extend(plot_sim_and_roc(pos_lvl, neg_lvl, level_title, output_dir))
            results.append(tmp_list)

    return results


In [ ]:
import os
import pandas as pd

_TIGRESS_SCHEMES = {
    "addopaque", "split", "merge", "addstack", "addvirtual",
    "encode", "initopaque", "reorder", "vm", "virtualize"
}
_GROUPS = ["tigress", "llvm-obfuscator"]

def _obf_group_from_path(path: str):
    """
    Classify an IDB path into 'tigress' or 'llvm-obfuscator'.
    Heuristic: look at hyphen-delimited tokens in the file stem.
      .../x64-clang-4.0.1-fla-xorriso.i64 -> 'llvm-obfuscator'
      .../x64-clang-14.0.0-addopaque-foo.i64 -> 'tigress'
    """
    if not isinstance(path, str):
        return None
    stem = os.path.splitext(os.path.basename(path))[0]
    for tok in stem.split('-'):
        if tok in _TIGRESS_SCHEMES:
            return "tigress"
    return "llvm-obfuscator"

def _add_obf_group_cols(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    if 'idb_path_1' in out.columns:
        out['group1'] = out['idb_path_1'].map(_obf_group_from_path)
    if 'idb_path_2' in out.columns:
        out['group2'] = out['idb_path_2'].map(_obf_group_from_path)
    return out

def compute_auc_by_obfuscator_and_plot(df_pos, df_neg, results_dir, output_dir):
    """
    Dataset-4, grouped by obfuscator:
      - Groups: 'tigress' (any of the listed Tigress passes) vs 'llvm-obfuscator' (everything else).
      - Keep ONLY pairs where both sides are in the SAME group.
      - For each CSV (model/test_name) and each group:
          * print the lowest-5 positive sims and highest-5 negative sims
          * call your plot_sim_and_roc(pos_slice, neg_slice, title, output_dir)
          * append a tmp_list to results starting with ["title", "..."] then extend with plot metrics
    Return value mirrors your original: a list of tmp_lists.
    """
    os.makedirs(output_dir, exist_ok=True)
    results = []

    for csv_file in sorted(os.listdir(results_dir)):
        if not csv_file.endswith(".csv") or "pos_testing" not in csv_file:
            continue

        print(
            "[D] Processing by obfuscator\n\t{}\n\t{}".format(
                csv_file,
                csv_file.replace("pos_testing", "neg_testing")
            )
        )
        test_name = csv_file.replace("pos_testing_", "").replace(".csv", "")

        # Load paired CSVs
        pos_path = os.path.join(results_dir, csv_file)
        neg_path = os.path.join(results_dir, csv_file.replace("pos_testing", "neg_testing"))
        df_pos_sim = pd.read_csv(pos_path, encoding="utf-8")
        df_neg_sim = pd.read_csv(neg_path, encoding="utf-8")

        # Sanity check
        assert df_pos_sim['sim'].isna().sum() == 0
        assert df_neg_sim['sim'].isna().sum() == 0

        # Merge with your metadata (same as your original function)
        df_pos_m = merge_data(df_pos, df_pos_sim, is_pos=True)
        df_neg_m = merge_data(df_neg, df_neg_sim, is_pos=False)

        # Tag obfuscator groups from paths
        df_pos_m = _add_obf_group_cols(df_pos_m)
        df_neg_m = _add_obf_group_cols(df_neg_m)

        # Per-group analysis; drop cross-group negatives
        for group in _GROUPS:
            pos_g = df_pos_m[(df_pos_m['group1'] == group) & (df_pos_m['group2'] == group)].copy()
            neg_g = df_neg_m[(df_neg_m['group1'] == group) & (df_neg_m['group2'] == group)].copy()

            if pos_g.empty or neg_g.empty:
                print(f"[W] Skipping '{test_name}' group {group}: "
                      f"{'no positives' if pos_g.empty else ''}"
                      f"{' and ' if pos_g.empty and neg_g.empty else ''}"
                      f"{'no negatives' if neg_g.empty else ''}.")
                continue

            # Diagnostics (mirrors your print style)
            lowest_pos = pos_g.nsmallest(5, 'sim')
            print(f"\n5 positive pairs with lowest sim for '{test_name}' [obf={group}]:")
            print(lowest_pos.to_string(index=False))
            print("-" * 80)

            highest_neg = neg_g.nlargest(5, 'sim')
            print(f"5 negative pairs with highest sim for '{test_name}' [obf={group}]:")
            print(highest_neg.to_string(index=False))
            print("=" * 100 + "\n")

            # Plots & metrics (reuse your utility)
            title = f"{test_name}_Obf_{group}"
            tmp_list = [["title", f"{test_name} [obf={group}]"]]
            tmp_list.extend(plot_sim_and_roc(pos_g, neg_g, title, output_dir))
            results.append(tmp_list)

    return results


In [ ]:
def from_list_to_df(auc_list):
    pd_temp_dict = defaultdict(list)
    for xr in auc_list:
        columns_set = set()
        columns = [x[0].strip() for x in xr]
        values = [x[1].strip() for x in xr]
        for c, v in zip(columns, values):
            columns_set.add(c)
            pd_temp_dict[c].append(v)
    df_auc = pd.DataFrame.from_dict(pd_temp_dict)
    df_auc = df_auc.rename(columns={"title":"model_name"})
    df_auc['model_name'] = df_auc['model_name'].apply(lambda x: x.replace("Dataset-1_", ""))
    df_auc['model_name'] = df_auc['model_name'].apply(lambda x: x.replace("Dataset-2_", ""))    
    df_auc['model_name'] = df_auc['model_name'].apply(lambda x: x.replace("Dataset-3_", ""))    
    df_auc['model_name'] = df_auc['model_name'].apply(lambda x: x.replace("Dataset-4_", ""))    
    df_auc['model_name'] = df_auc['model_name'].apply(lambda x: x.replace("Dataset-5_", ""))    
    df_auc['model_name'] = df_auc['model_name'].apply(lambda x: x.replace("Dataset-6_", ""))    
    df_auc['model_name'] = df_auc['model_name'].apply(lambda x: x.replace("Dataset-7_", ""))    
    return df_auc

In [ ]:
# Create output folders
!mkdir -p metrics_and_plots/Dataset-1
!mkdir -p metrics_and_plots/Dataset-2
!mkdir -p metrics_and_plots/Dataset-3
!mkdir -p metrics_and_plots/Dataset-4
!mkdir -p metrics_and_plots/Dataset-5
!mkdir -p metrics_and_plots/Dataset-6
!mkdir -p metrics_and_plots/Dataset-7

## Dataset 1

In [ ]:
RESULTS_DIR = "../data/Dataset-1/"
OUTPUT_DIR = "metrics_and_plots/Dataset-1/"

base_path = "../../DBs/Dataset-1/pairs/testing/"

df_pos_testing = pd.read_csv(
    os.path.join(base_path, "pos_testing_Dataset-1.csv"))

df_neg_testing = pd.read_csv(
    os.path.join(base_path, "neg_testing_Dataset-1.csv"))

auc_list = compute_auc_and_plot(df_pos_testing, df_neg_testing, RESULTS_DIR, OUTPUT_DIR)
df_auc = from_list_to_df(auc_list)    
# display(df_auc)
df_auc.to_csv(os.path.join(OUTPUT_DIR, "df_auc.csv"))

## Dataset 2

In [ ]:
RESULTS_DIR = "../data/Dataset-2/"
OUTPUT_DIR = "metrics_and_plots/Dataset-2/"

base_path = "../../DBs/Dataset-2/pairs/testing/"

df_pos_testing = pd.read_csv(
    os.path.join(base_path, "pos_testing_Dataset-2.csv"))

df_neg_testing = pd.read_csv(
    os.path.join(base_path, "neg_testing_Dataset-2.csv"))

auc_list = compute_auc_and_plot(df_pos_testing, df_neg_testing, RESULTS_DIR, OUTPUT_DIR)
df_auc = from_list_to_df(auc_list)    
display(df_auc)
df_auc.to_csv(os.path.join(OUTPUT_DIR, "df_auc.csv"))

## Dataset 3

In [ ]:
RESULTS_DIR = "../data/Dataset-3/"
OUTPUT_DIR = "metrics_and_plots/Dataset-3/"

base_path = "../../DBs/Dataset-3/pairs/testing/"

df_pos_testing = pd.read_csv(
    os.path.join(base_path, "pos_testing_Dataset-3.csv"))

df_neg_testing = pd.read_csv(
    os.path.join(base_path, "neg_testing_Dataset-3.csv"))

auc_list = compute_auc_and_plot(df_pos_testing, df_neg_testing, RESULTS_DIR, OUTPUT_DIR)
df_auc = from_list_to_df(auc_list)    
display(df_auc)
df_auc.to_csv(os.path.join(OUTPUT_DIR, "df_auc.csv"))

## Dataset 4

In [ ]:
RESULTS_DIR = "../data/Dataset-4/"
OUTPUT_DIR = "metrics_and_plots/Dataset-4/"

base_path = "../../DBs/Dataset-4/pairs/testing/"

df_pos_testing = pd.read_csv(
    os.path.join(base_path, "pos_testing_Dataset-4.csv"))
df_neg_testing = pd.read_csv(
    os.path.join(base_path, "neg_testing_Dataset-4.csv"))

# ─────────────────────────────────────────────────────────
# Original analysis (overall)
auc_list = compute_auc_and_plot(df_pos_testing, df_neg_testing, RESULTS_DIR, OUTPUT_DIR)
df_auc = from_list_to_df(auc_list)    
display(df_auc)
df_auc.to_csv(os.path.join(OUTPUT_DIR, "df_auc.csv"), index=False)

# ─────────────────────────────────────────────────────────
# New analysis (by obfuscator group)
auc_list_obf = compute_auc_by_obfuscator_and_plot(
    df_pos_testing, df_neg_testing, RESULTS_DIR, OUTPUT_DIR
)
df_auc_obf = from_list_to_df(auc_list_obf)
display(df_auc_obf)
df_auc_obf.to_csv(os.path.join(OUTPUT_DIR, "df_auc_obfuscator.csv"), index=False)


## Dataset 5

In [ ]:
RESULTS_DIR = "../data/Dataset-5/"
OUTPUT_DIR = "metrics_and_plots/Dataset-5/"

base_path = "../../DBs/Dataset-5/pairs/testing/"

df_pos_testing = pd.read_csv(
    os.path.join(base_path, "pos_testing_Dataset-5.csv"))
df_neg_testing = pd.read_csv(
    os.path.join(base_path, "neg_testing_Dataset-5.csv"))

# ─────────────────────────────────────────────────────────
# Original per-model analysis
auc_list = compute_auc_and_plot(df_pos_testing, df_neg_testing, RESULTS_DIR, OUTPUT_DIR)
df_auc = from_list_to_df(auc_list)    
display(df_auc)
df_auc.to_csv(os.path.join(OUTPUT_DIR, "df_auc.csv"), index=False)

# ─────────────────────────────────────────────────────────
# New per-difficulty analysis
auc_list_difficulty = compute_auc_by_difficulty_and_plot(
    df_pos_testing, df_neg_testing, RESULTS_DIR, OUTPUT_DIR
)
df_auc_difficulty = from_list_to_df(auc_list_difficulty)
display(df_auc_difficulty)
df_auc_difficulty.to_csv(os.path.join(OUTPUT_DIR, "df_auc_difficulty.csv"), index=False)


## Dataset 6

In [ ]:
RESULTS_DIR = "../data/Dataset-6/"
OUTPUT_DIR = "metrics_and_plots/Dataset-6/"

base_path = "../../DBs/Dataset-6/pairs/testing/"

df_pos_testing = pd.read_csv(
    os.path.join(base_path, "pos_testing_Dataset-6.csv"))

df_neg_testing = pd.read_csv(
    os.path.join(base_path, "neg_testing_Dataset-6.csv"))


auc_list = compute_auc_and_plot(df_pos_testing, df_neg_testing, RESULTS_DIR, OUTPUT_DIR)
df_auc = from_list_to_df(auc_list)    
display(df_auc)
df_auc.to_csv(os.path.join(OUTPUT_DIR, "df_auc.csv"))

## Dataset 7

In [ ]:
RESULTS_DIR = "../data/Dataset-7/"
OUTPUT_DIR = "metrics_and_plots/Dataset-7/"

base_path = "../../DBs/Dataset-7/pairs/testing/"

df_pos_testing = pd.read_csv(
    os.path.join(base_path, "pos_testing_Dataset-7.csv"))

df_neg_testing = pd.read_csv(
    os.path.join(base_path, "neg_testing_Dataset-7.csv"))


auc_list = compute_auc_and_plot(df_pos_testing, df_neg_testing, RESULTS_DIR, OUTPUT_DIR)
df_auc = from_list_to_df(auc_list)    
display(df_auc)
df_auc.to_csv(os.path.join(OUTPUT_DIR, "df_auc.csv"))